# Phutball expert iteration: independent vs strategically diverse sampling (Qwen3.5-9B, thinking on)

Each round samples k=4 solutions per training puzzle with the current model, keeps the model's own **engine-verified**
solutions (a run-on sample is kept only up to its shortest stably verified point), and fine-tunes from the base model on
them (ReST-EM). Only verified answers are ever trained on, so the "NO WIN" collapse of the GRPO run cannot happen.

Two arms, same budget:
- **iid**: 4 independent samples per puzzle.
- **strategic** (Gurung, Whitammer & Lapata, *Strategically Diverse Sampling for Self-Training*): the model lists 4
  different strategies, solves once per strategy with it as a hidden instruction; training keeps only puzzle -> solution.

Evals are sampled (temperature 0.6 / top-p 0.95 / top-k 20, never greedy in thinking mode), 4 samples per item on 275
benchmark items: **pass@1** (the gate) and **pass@4** (coverage, which decides whether RL gets signal), strict and budget-forced.

**Runtime:** A100 GPU, High-RAM. **Cost:** baselines ~1 h, each round ~2.5 h for both arms incl. evals (~17 units).
Everything is skipped if already in Drive, so Run all resumes. **Disconnect the runtime when done.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, subprocess, time, urllib.request
OUT = '/content/drive/MyDrive/phutball/expert_iter'; os.makedirs(OUT, exist_ok=True)
WARM = '/content/drive/MyDrive/phutball/grpo_pilot/self_stop/adapter/final'   # round 0: the self-stop warm start
CAP = 6144; N = 600; K = 4; PG = 25; EK = 4; ROUNDS = 1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
def serve(adapters: dict, max_len=CAP + 2048):
    """vLLM with the base model plus named LoRA adapters ({name: path}); the base model is 'Qwen/Qwen3.5-9B'."""
    !pkill -f "vllm serve" ; sleep 5
    extra = ['--enable-lora', '--max-lora-rank', '32', '--max-loras', str(max(1, len(adapters))), '--lora-modules',
             *[f'{k}={v}' for k, v in adapters.items()]] if adapters else []
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(max_len), '--gpu-memory-utilization',
                          '0.90', '--port', '8000', *extra], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if p.poll() is not None: break
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', list(adapters) or 'base'); return p
        except Exception: time.sleep(5)
    !tail -15 /content/vllm_serve.log
    raise RuntimeError('vLLM did not start')

EV = f'{OUT}/eval'; os.makedirs(EV, exist_ok=True)
def evaluate(name, model):
    """Sampled eval, EK samples per item: strict (finished) and budget-forced summaries -> EV/name*."""
    if os.path.exists(f'{EV}/{name}-forced.summary.json'): print('done:', name); return
    !python -m llm_bench.run --model {model} --name {name} --thinking true --qwen-thinking-sampling --samples {EK} --max-tokens {CAP} --per-group {PG} --concurrency 96 | tail -10
    !python -m llm_bench.run --model {model} --tokenizer Qwen/Qwen3.5-9B --thinking true --salvage {name} --per-group {PG} --concurrency 48 | tail -10
    !cp llm_bench/results/{name}* {EV}/

## 1. Baselines: untrained and warm start (sampled, the yardstick for every round)

In [ ]:
if not all(os.path.exists(f'{EV}/{n}-forced.summary.json') for n in ('untrained', 'warmstart')):
    server = serve({'warmstart': WARM})
    evaluate('untrained', 'Qwen/Qwen3.5-9B')
    evaluate('warmstart', 'warmstart')
    !pkill -f "vllm serve"
else: print('baselines done')

## 2. Rounds (both arms; each arm continues from its own previous round)

In [ ]:
prev = {'iid': WARM, 'strategic': WARM}
for R in range(1, ROUNDS + 1):
    for arm in ('iid', 'strategic'):
        d = f'{OUT}/r{R}_{arm}'
        if not os.path.exists(f'{d}/sft.jsonl'):
            print(f'=== round {R} {arm}: sampling from', prev[arm])
            !python -m llm_train.expert_iter --arm {arm} --adapter {prev[arm]} --out {d} --n {N} --k {K} --max-think {CAP} --seed {R} 2>&1 | grep --line-buffered -v -E "it/s\]|Processed prompts|Adding requests|INFO|WARNING"
        if not os.path.exists(f'{d}/adapter/final/adapter_model.safetensors'):
            !python -m llm_train.sft_stop --data {d}/sft.jsonl --out {d}/adapter 2>&1 | grep --line-buffered -E "examples|'loss'|saved|rror|Traceback"
        prev[arm] = f'{d}/adapter/final'
    todo = {f'r{R}_{arm}': prev[arm] for arm in ('iid', 'strategic') if not os.path.exists(f'{EV}/r{R}_{arm}-forced.summary.json')}
    if todo:
        server = serve(todo)
        for name in todo: evaluate(name, name)
        !pkill -f "vllm serve"
print('rounds done; latest:', prev)

## 3. Results

In [ ]:
names = ['untrained', 'warmstart'] + [f'r{R}_{arm}' for R in range(1, ROUNDS + 1) for arm in ('iid', 'strategic')]
keys = ('win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump', 'win: positive, 4-jump',
        'win: near-miss negative', 'forced placement', 'block placement', 'prevent placement')
for label, suffix in (('FINISHED ON ITS OWN', '.summary.json'), ('BUDGET-FORCED', '-forced.summary.json')):
    S = {n: json.load(open(f'{EV}/{n}{suffix}')) for n in names if os.path.exists(f'{EV}/{n}{suffix}')}
    print(f"\n{label}: pass@1 [95% CI] / pass@{EK}  (thinking on, sampled, {CAP} tokens; pass@{EK} is trivial on no-win items)")
    print(f"{'':26s}" + ''.join(f"{n:>24s}" for n in S))
    for k in keys:
        row = []
        for n in S:
            g = S[n].get(k)
            row.append(f"{g['accuracy']:.0%} [{g['ci95'][0]:.0%}-{g['ci95'][1]:.0%}] / {g.get(f'pass@{EK}', float('nan')):.0%}" if g else '-')
        print(f"{k:26s}" + ''.join(f"{c:>24s}" for c in row))
    print(f"{'cut off (wins)':26s}" + ''.join(f"{S[n].get('win (all)', {}).get('truncated', 0):>24.0%}" for n in S))
    print(f"{'mean completion tokens':26s}" + ''.join(f"{S[n].get('mean_completion_tokens', 0):>24.0f}" for n in S))
for R in range(1, ROUNDS + 1):
    for arm in ('iid', 'strategic'):
        f = f'{OUT}/r{R}_{arm}/stats.json'
        if os.path.exists(f):
            st = json.load(open(f)); print(f"\nr{R}_{arm}: {st['examples']} examples, solved {st['puzzles_with_a_verified_solution']}, "
                                           f"median {st['median_think_tokens']} thinking tokens, strategies {st['distinct_strategies_kept']}, {st['sample_outcomes']}")